# Genotype–Phenotype Heterogeneity and Infertility Management Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset package using the `mlcroissant` library. All entities are referenced by their `@id` fields for clarity and consistency.

### Dataset Source
The dataset is defined via a Croissant schema at:

[https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Single metadata object
print(f"Dataset title: {metadata.name}\n\n{metadata.description}")

# Show data collection, timeframe, keywords
print("\nData Collection:", metadata.dataCollection)
print("Keywords:", metadata.keywords)


## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities are referenced by their `@id`.

In [ ]:
# List all RecordSets and their @id
record_sets = dataset.record_sets

print("Available RecordSets:")
for rs in record_sets:
    print(f" - @id: {rs['@id']}, name: {rs.get('name','')}")

# List fields and columns for each RecordSet by @id
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']} - Fields:")
    for field in rs.get('field', []):
        print(f"   Field @id: {field['@id']} | name: {field.get('name','')} | dataType: {field.get('dataType','')}")
        if 'column' in field:
            for col in field['column']:
                print(f"     Column @id: {col['@id']} | name: {col.get('name','')} | dataType: {col.get('dataType','')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references use `@id`.

In [ ]:
# Extract all record sets (by @id)
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for RecordSet {record_set_id}:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply typical processing steps using columns referenced by their `@id`. Filter and normalize numeric fields, handle grouping and outliers.

In [ ]:
# Example: Choose a RecordSet and numeric field by @id
# For illustration, let's work with the first record set
example_record_set_id = record_set_ids[0]
df = dataframes[example_record_set_id]

# Find a numeric field's @id
numeric_field_id = None
for field in dataset.record_sets[0].get('field', []):
    if field.get('dataType') in ('schema:Number', 'schema:Integer', 'schema:Float'):
        numeric_field_id = field['@id']
        break

if numeric_field_id and numeric_field_id in df.columns:
    # Filtering numeric field
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field
    group_field_id = None
    for field in dataset.record_sets[0].get('field', []):
        if field.get('dataType') == 'schema:Text':
            group_field_id = field['@id']
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA in this RecordSet.")

## 5. Visualization
Visualize distributions or relationships between fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution if found
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()


## 6. Conclusion
* Using the `mlcroissant` library, we loaded the FAIR^2 dataset, extracted structured tabular data, and referenced entities exclusively by their `@id`.
* We performed typical EDA steps: filtering, normalization, and grouping, and produced field-level visualizations.
* The small, specialized clinical cohort illustrates genotype-phenotype variability but is not suitable for predictive AI modeling without further expansion.
* All processing steps can be adapted for any Croissant-encoded FAIR dataset by updating `@id` references as needed.